# Annotate an RO-Crate with structured metadata

This notebook enriches an existing RO-Crate in ROHub with three layers of metadata:

1. **Extracted from data** — read CF attributes from Zarr (variables, coverage, resolution)
2. **I-ADOPT variable decomposition** — decompose variable names into Property + ObjectOfInterest + Matrix, publish as nanopublications
3. **Expert.AI claim extraction** — extract key claims from the dataset description

ROHub also auto-enriches with Expert.AI topic classification (layer 4), which happens automatically.

### Prerequisites

- `~/rohub_credentials.json`
- I-ADOPT service running locally (`docker compose up` in i-adopt-service-website)
- Internet access (for Expert.AI claim API and ROHub)

In [ ]:
import rohub
import json
import requests
import fsspec
import datetime
import uuid
from pathlib import Path
from rohub import settings, utils

## Configuration

Set the RO-Crate to annotate. This notebook works with any of the three FAIR2Adapt examples.

In [ ]:
# Choose one:
ro_pid = "https://w3id.org/ro-id/24600867-23ca-4b97-be14-3aa63883c056"  # RIOMAR
# ro_pid = "https://w3id.org/ro-id/fdc1c071-76d7-44df-a565-8217ebcc59fe"  # Sentinel-2
# ro_pid = "https://w3id.org/ro-id/1f0b5044-ae4f-483d-b7a2-48a5a6ac3965"  # NorESM2 ARCTIC

# Service endpoints
IADOPT_URL = "http://localhost:8000"  # I-ADOPT decomposition service
CLAIM_URL = "https://labdemos.expertcustomers.ai/services/claim_extraction"  # Expert.AI

# schema.org and I-ADOPT vocabulary
SCHEMA = "https://schema.org/"
DCTERMS = "http://purl.org/dc/terms/"
IADOPT = "https://w3id.org/iadopt/ont/"
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"

ro_id = ro_pid.rstrip("/").split("/")[-1]
print(f"RO-Crate: {ro_pid}")

## Login to ROHub and load the RO-Crate

In [ ]:
credentials = json.loads(Path("~/rohub_credentials.json").expanduser().read_text())
rohub.login(username=credentials["username"], password=credentials["password"])

ro = rohub.ros_load(identifier=ro_id)
print(f"Title: {ro.title}")
print(f"Description: {ro.description[:200]}...")

resources = ro.list_resources()
print(f"\nResources:")
print(resources[["identifier", "type", "name", "url"]].to_string())

# Get the dataset URL
dataset = resources[resources["type"] == "Dataset"].iloc[0]
dataset_url = dataset["url"]
dataset_res = f"{ro_pid}/resources/{dataset['identifier']}"
print(f"\nDataset URL: {dataset_url}")

## Layer 1: Extract metadata from the data

Read CF attributes directly from the Zarr store to discover variables,
spatial/temporal coverage, and resolution. This is what the data creator
provided — the foundation for all other metadata layers.

In [ ]:
# Read Zarr metadata (works with both public and proxy URLs)
zarr_url = dataset_url.rstrip("/")

# Try direct access, then with sublevels (for multiscale pyramids)
mapper = fsspec.get_mapper(zarr_url)
root_meta = json.loads(mapper["zarr.json"])
root_attrs = root_meta.get("attributes", {})

# Check for multiscale pyramid
multiscales = root_attrs.get("multiscales")
if multiscales:
    finest = multiscales["layout"][0]["asset"]
    var_meta = json.loads(mapper[f"{finest}/zarr.json"]).get("consolidated_metadata", {}).get("metadata", {})
    print(f"Multiscale pyramid detected, finest level: {finest}")
else:
    var_meta = root_meta.get("consolidated_metadata", {}).get("metadata", {})

print(f"\nRoot attributes: {list(root_attrs.keys())}")
print(f"Variables/arrays: {list(var_meta.keys())}")

In [ ]:
# Identify data variables (exclude coordinates and grid variables)
COORDINATE_NAMES = {
    "cell", "cell_ids", "crs", "time", "time_counter", "time_instant",
    "s_rho", "sigma", "depth", "depth_bnds", "latitude", "longitude",
    "lat", "lon", "bounds",
}

data_vars = {}
for vname, vmeta in var_meta.items():
    if vname in COORDINATE_NAMES:
        continue
    attrs = vmeta.get("attributes", {})
    if "long_name" in attrs or "standard_name" in attrs:
        data_vars[vname] = {
            "long_name": attrs.get("long_name", vname),
            "standard_name": attrs.get("standard_name"),
            "units": attrs.get("units", "dimensionless"),
            "shape": vmeta.get("shape", []),
        }

print("Data variables found:")
for vname, info in data_vars.items():
    print(f"  {vname}: {info['long_name']} [{info['units']}]")
    if info['standard_name']:
        print(f"    CF standard_name: {info['standard_name']}")

In [ ]:
# Extract grid information
grid_info = {}

# Check for HEALPix
for vname in ["cell", "cell_ids", "crs"]:
    if vname in var_meta:
        attrs = var_meta[vname].get("attributes", {})
        if attrs.get("grid_name") == "healpix" or attrs.get("grid_mapping_name") == "healpix":
            level = attrs.get("level") or attrs.get("healpix_nside")
            nside = attrs.get("healpix_nside", 2 ** attrs.get("level", 0))
            grid_info = {
                "type": "HEALPix",
                "level": attrs.get("level"),
                "nside": nside,
                "indexing": attrs.get("indexing_scheme", attrs.get("healpix_order", "nested")),
            }
            break

# Check for curvilinear (2D lat/lon)
if not grid_info:
    for vname in ["latitude", "lat"]:
        if vname in var_meta and len(var_meta[vname].get("shape", [])) == 2:
            grid_info = {"type": "curvilinear"}
            break

if not grid_info:
    grid_info = {"type": "regular"}

# Resolution text
if grid_info["type"] == "HEALPix" and multiscales:
    levels = [json.loads(mapper[f"{l['asset']}/zarr.json"]).get("consolidated_metadata", {}).get("metadata", {}) 
              for l in [multiscales["layout"][0], multiscales["layout"][-1]]]
    finest_level = levels[0].get("cell", levels[0].get("cell_ids", {})).get("attributes", {}).get("level", "?")
    coarsest_level = levels[1].get("cell", levels[1].get("cell_ids", {})).get("attributes", {}).get("level", "?")
    resolution_text = f"HEALPix multiscale pyramid, level {finest_level} to level {coarsest_level}"
elif grid_info["type"] == "HEALPix":
    resolution_text = f"HEALPix level {grid_info['level']} (nside={grid_info['nside']})"
else:
    resolution_text = f"{grid_info['type']} grid"

print(f"Grid: {grid_info}")
print(f"Resolution: {resolution_text}")

## Layer 2: I-ADOPT variable decomposition

For each data variable, we call the I-ADOPT service to decompose the
`long_name` into structured components (Property, ObjectOfInterest, Matrix)
and enrich with Wikidata URIs. The results can be published as nanopublications.

This step requires the I-ADOPT service running locally:
```bash
cd i-adopt-service-website && docker compose up
```

In [ ]:
# Check if I-ADOPT service is available
try:
    r = requests.get(f"{IADOPT_URL}/health", timeout=5)
    iadopt_available = r.status_code == 200
except Exception:
    iadopt_available = False

print(f"I-ADOPT service: {'available' if iadopt_available else 'NOT RUNNING — skipping decomposition'}")

In [ ]:
decompositions = {}

if iadopt_available:
    for vname, info in data_vars.items():
        definition = f"{info['long_name']}"
        if info['units'] != 'dimensionless':
            definition += f" in {info['units']}"
        
        print(f"\nDecomposing: {definition}")
        
        resp = requests.post(
            f"{IADOPT_URL}/decompose",
            json={"definition": definition},
            timeout=60,
        )
        
        if resp.ok:
            result = resp.json()
            decompositions[vname] = result
            parsed = result.get("enriched_json") or result.get("parsed_json", {})
            print(f"  Property: {parsed.get('hasProperty', '?')}")
            print(f"  ObjectOfInterest: {parsed.get('hasObjectOfInterest', '?')}")
            print(f"  Matrix: {parsed.get('hasMatrix', '?')}")
            if result.get("ttl"):
                print(f"  TTL: {len(result['ttl'])} chars")
        else:
            print(f"  ERROR: {resp.status_code} {resp.text[:200]}")
else:
    print("Skipping I-ADOPT decomposition (service not running)")

In [ ]:
# Save decompositions locally for reuse
if decompositions:
    cache_file = Path(f"iadopt_cache_{ro_id[:8]}.json")
    cache_file.write_text(json.dumps(decompositions, indent=2))
    print(f"Saved {len(decompositions)} decompositions to {cache_file}")
    
    # Show TTL for the first variable
    first = list(decompositions.values())[0]
    if first.get("ttl"):
        print(f"\nExample TTL for '{list(decompositions.keys())[0]}':")
        print(first["ttl"][:500])

## Layer 3: Expert.AI claim extraction

Extract key claims from the dataset description using the Expert.AI service.
These claims capture the main statements about what the data shows or provides.

In [ ]:
description = ro.description
print(f"Extracting claims from description ({len(description)} chars)...")
print(f"\n\"{description[:300]}...\"\n")

claims = []
try:
    resp = requests.post(
        CLAIM_URL,
        headers={"Content-Type": "text/plain"},
        data=description,
        timeout=30,
    )
    if resp.ok:
        raw_claims = resp.json()
        # Deduplicate
        seen = set()
        for c in raw_claims:
            text = c.get("claim", "").strip()
            if text and text.lower() not in seen:
                claims.append(text)
                seen.add(text.lower())
        print(f"Extracted {len(claims)} claims:")
        for i, c in enumerate(claims, 1):
            print(f"  {i}. {c}")
    else:
        print(f"Expert.AI returned {resp.status_code}")
except Exception as e:
    print(f"Expert.AI service error: {e}")

## Add annotations to the RO-Crate

Now we combine all three layers into structured annotations on the RO-Crate,
following the pattern from `riomar_rocrate.ipynb`.

In [ ]:
# Reload RO and create a new annotation
ro = rohub.ros_load(identifier=ro_id)
annot = ro.add_annotations()
aid = annot["identifier"]
print(f"Created annotation: {aid}")

# Product ID for this dataset
mvp_id = f"{ro_pid}/product/{uuid.uuid4().hex[:8]}"

In [ ]:
# Link product to RO-Crate
ro.add_triple(
    the_subject=ro_pid, the_predicate=f"{SCHEMA}isRelatedTo",
    the_object=mvp_id, annotation_id=aid, object_class="URIRef",
)
ro.add_triple(
    the_subject=mvp_id, the_predicate=RDF_TYPE,
    the_object=f"{SCHEMA}Product", annotation_id=aid, object_class="URIRef",
)

# Resolution
ro.add_triple(
    the_subject=mvp_id,
    the_predicate="http://data.europa.eu/930/spatialResolutionAsText",
    the_object=resolution_text, annotation_id=aid,
)

# Identifier
ro.add_triple(
    the_subject=mvp_id, the_predicate=f"{SCHEMA}identifier",
    the_object=dataset_url, annotation_id=aid,
)

# Provenance
ro.add_triple(
    the_subject=mvp_id, the_predicate=f"{DCTERMS}provenance",
    the_object="https://fair2adapt.github.io/riomar-dashboard/",
    annotation_id=aid, object_class="URIRef",
)

print(f"Product: {mvp_id}")
print(f"Resolution: {resolution_text}")

In [ ]:
# Add variables with I-ADOPT decomposition
for i, (vname, info) in enumerate(data_vars.items(), 1):
    var_id = f"{ro_pid}/variable/{i}"
    
    # Link variable to product
    ro.add_triple(
        the_subject=mvp_id, the_predicate=f"{SCHEMA}variableMeasured",
        the_object=var_id, annotation_id=aid, object_class="URIRef",
    )
    ro.add_triple(
        the_subject=var_id, the_predicate=RDF_TYPE,
        the_object=f"{IADOPT}Variable", annotation_id=aid, object_class="URIRef",
    )
    
    # Variable name
    var_label = info["long_name"]
    ro.add_triple(
        the_subject=var_id, the_predicate=f"{SCHEMA}name",
        the_object=var_label, annotation_id=aid,
    )
    
    # Add I-ADOPT decomposition if available
    decomp = decompositions.get(vname)
    if decomp:
        parsed = decomp.get("enriched_json") or decomp.get("parsed_json", {})
        
        # Property
        if parsed.get("hasProperty"):
            prop_id = f"{var_id}/property"
            ro.add_triple(
                the_subject=var_id, the_predicate=f"{IADOPT}hasProperty",
                the_object=prop_id, annotation_id=aid, object_class="URIRef",
            )
            ro.add_triple(
                the_subject=prop_id, the_predicate=f"{SCHEMA}name",
                the_object=parsed["hasProperty"], annotation_id=aid,
            )
            if parsed.get("hasPropertyURI"):
                ro.add_triple(
                    the_subject=prop_id, the_predicate=f"{SCHEMA}url",
                    the_object=parsed["hasPropertyURI"], annotation_id=aid, object_class="URIRef",
                )
        
        # ObjectOfInterest
        ooi = parsed.get("hasObjectOfInterest")
        if ooi and isinstance(ooi, str):
            ooi_id = f"{var_id}/entity"
            ro.add_triple(
                the_subject=var_id, the_predicate=f"{IADOPT}hasObjectOfInterest",
                the_object=ooi_id, annotation_id=aid, object_class="URIRef",
            )
            ro.add_triple(
                the_subject=ooi_id, the_predicate=f"{SCHEMA}name",
                the_object=ooi, annotation_id=aid,
            )
            if parsed.get("hasObjectOfInterestURI"):
                ro.add_triple(
                    the_subject=ooi_id, the_predicate=f"{SCHEMA}url",
                    the_object=parsed["hasObjectOfInterestURI"], annotation_id=aid, object_class="URIRef",
                )
        
        # Matrix
        matrix = parsed.get("hasMatrix")
        if matrix and isinstance(matrix, str):
            matrix_id = f"{var_id}/matrix"
            ro.add_triple(
                the_subject=var_id, the_predicate=f"{IADOPT}hasMatrix",
                the_object=matrix_id, annotation_id=aid, object_class="URIRef",
            )
            ro.add_triple(
                the_subject=matrix_id, the_predicate=f"{SCHEMA}name",
                the_object=matrix, annotation_id=aid,
            )
        
        # Link to TTL/nanopub if available
        if decomp.get("ttl"):
            ro.add_triple(
                the_subject=var_id, the_predicate=f"{DCTERMS}description",
                the_object=f"I-ADOPT decomposition: {parsed.get('hasProperty', '?')} of {ooi or '?'}",
                annotation_id=aid,
            )
    
    print(f"  {vname}: {var_label}")
    if decomp:
        parsed = decomp.get("enriched_json") or decomp.get("parsed_json", {})
        print(f"    I-ADOPT: {parsed.get('hasProperty', '?')} | {parsed.get('hasObjectOfInterest', '?')} | {parsed.get('hasMatrix', '?')}")

In [ ]:
# Add Expert.AI claims
if claims:
    for i, claim_text in enumerate(claims, 1):
        claim_id = f"{ro_pid}/claim/{i}"
        ro.add_triple(
            the_subject=dataset_res, the_predicate=f"{SCHEMA}hasClaim",
            the_object=claim_id, annotation_id=aid, object_class="URIRef",
        )
        ro.add_triple(
            the_subject=claim_id, the_predicate=RDF_TYPE,
            the_object=f"{SCHEMA}Claim", annotation_id=aid, object_class="URIRef",
        )
        ro.add_triple(
            the_subject=claim_id, the_predicate=f"{SCHEMA}text",
            the_object=claim_text, annotation_id=aid,
        )
        print(f"  Claim {i}: {claim_text[:80]}..." if len(claim_text) > 80 else f"  Claim {i}: {claim_text}")
else:
    print("No claims to add")

## Verify

In [ ]:
print(f"RO-Crate: {ro_pid}")
print(f"Annotation: {aid}")
print(f"\nSummary:")
print(f"  Variables: {len(data_vars)}")
print(f"  I-ADOPT decompositions: {len(decompositions)}")
print(f"  Expert.AI claims: {len(claims)}")
print(f"  Grid: {resolution_text}")
print(f"\nView in ROHub: https://reliance.rohub.org/explore/{ro_id}")
print(f"Test with FDO2map: python fdo2map.py {ro_pid}")